# Forces and Fields: Discrete Operators

**"From flux gradients to the four fundamental forces."**

This notebook explores how forces emerge from discrete differential operators applied to the flux field.

---

## The Force Hierarchy

| Force | FTD Implementation | Physical Analog |
|-------|-------------------|------------------|
| Gravity | F = G_N ∇ρ | Attraction to mass density |
| Electric | F = -q ∇q_field | Coulomb force |
| Magnetic | F = β (∇×J) × Ĵ | Lorentz force |
| Strong | F = Yukawa form | Nuclear binding |

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

repo_root = os.path.abspath("../../")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from ternary_matrix.model.grid import Universe
from ternary_matrix.physics import master_equation, waves, forces
from ternary_matrix.config import CONSTANTS

print("Modules loaded.")

## 1. Discrete Differential Operators

FTD uses **finite difference** approximations for calculus on the lattice.

### Gradient (∇)
$$\nabla f(v) = \frac{f(v + e_i) - f(v - e_i)}{2}$$

### Divergence (∇·)
$$\nabla \cdot J = \sum_i \frac{J_i(v + e_i) - J_i(v - e_i)}{2}$$

### Curl (∇×)
$$\nabla \times J = \varepsilon_{ijk} \frac{\partial J_k}{\partial x_j}$$

### Laplacian (∇²)
$$\nabla^2 f(v) = \sum_{u \in N_6(v)} f(u) - 6f(v)$$

In [ ]:
# Implement and visualize discrete operators
def discrete_gradient(field):
    """Compute discrete gradient of a scalar field."""
    grad = np.zeros((*field.shape, 3))
    
    # Central differences
    grad[1:-1, :, :, 0] = (field[2:, :, :] - field[:-2, :, :]) / 2  # dF/dx
    grad[:, 1:-1, :, 1] = (field[:, 2:, :] - field[:, :-2, :]) / 2  # dF/dy
    grad[:, :, 1:-1, 2] = (field[:, :, 2:] - field[:, :, :-2]) / 2  # dF/dz
    
    return grad

def discrete_divergence(J):
    """Compute discrete divergence of a vector field."""
    div = np.zeros(J.shape[:-1])
    
    div[1:-1, :, :] += (J[2:, :, :, 0] - J[:-2, :, :, 0]) / 2
    div[:, 1:-1, :] += (J[:, 2:, :, 1] - J[:, :-2, :, 1]) / 2
    div[:, :, 1:-1] += (J[:, :, 2:, 2] - J[:, :, :-2, 2]) / 2
    
    return div

def discrete_curl(J):
    """Compute discrete curl of a vector field."""
    curl = np.zeros_like(J)
    
    # curl_x = dJz/dy - dJy/dz
    curl[:, 1:-1, :, 0] += (J[:, 2:, :, 2] - J[:, :-2, :, 2]) / 2
    curl[:, :, 1:-1, 0] -= (J[:, :, 2:, 1] - J[:, :, :-2, 1]) / 2
    
    # curl_y = dJx/dz - dJz/dx
    curl[:, :, 1:-1, 1] += (J[:, :, 2:, 0] - J[:, :, :-2, 0]) / 2
    curl[1:-1, :, :, 1] -= (J[2:, :, :, 2] - J[:-2, :, :, 2]) / 2
    
    # curl_z = dJy/dx - dJx/dy
    curl[1:-1, :, :, 2] += (J[2:, :, :, 1] - J[:-2, :, :, 1]) / 2
    curl[:, 1:-1, :, 2] -= (J[:, 2:, :, 0] - J[:, :-2, :, 0]) / 2
    
    return curl

In [ ]:
# Create test fields
size = 32
x, y, z = np.meshgrid(np.arange(size), np.arange(size), np.arange(size), indexing='ij')
center = size // 2

# Gaussian scalar field (like a mass distribution)
r = np.sqrt((x - center)**2 + (y - center)**2 + (z - center)**2)
scalar_field = np.exp(-r**2 / 50)

# Compute gradient
grad = discrete_gradient(scalar_field)

In [ ]:
# Visualize gradient
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
z_slice = center

# Scalar field
im0 = axes[0].imshow(scalar_field[:, :, z_slice].T, origin='lower', cmap='hot')
axes[0].set_title('Scalar Field (Gaussian)', fontsize=12)
plt.colorbar(im0, ax=axes[0])

# Gradient magnitude
grad_mag = np.linalg.norm(grad[:, :, z_slice, :], axis=-1)
im1 = axes[1].imshow(grad_mag.T, origin='lower', cmap='viridis')
axes[1].set_title('|∇f| (Gradient Magnitude)', fontsize=12)
plt.colorbar(im1, ax=axes[1])

# Gradient vectors
step = 2
X, Y = np.meshgrid(np.arange(0, size, step), np.arange(0, size, step))
Gx = grad[::step, ::step, z_slice, 0]
Gy = grad[::step, ::step, z_slice, 1]
axes[2].quiver(X, Y, Gx.T, Gy.T, np.sqrt(Gx**2 + Gy**2).T, cmap='plasma')
axes[2].set_title('∇f (Gradient Vectors)', fontsize=12)
axes[2].set_xlim(0, size)
axes[2].set_ylim(0, size)
axes[2].set_aspect('equal')

plt.suptitle('Discrete Gradient Operator', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Gravity-Like Force

Gravity in FTD comes from the gradient of the **smoothed density field**:

$$F_{grav} = G_N \cdot \nabla \bar{\rho}$$

This produces attraction toward high-density regions.

In [ ]:
# Set up a two-body gravitational scenario
CONSTANTS.C = 0.5
CONSTANTS.KB = 10.0  # Prevent manifestation
CONSTANTS.GRAVITY_BIAS = 0.05

universe = Universe(size=48)
center = universe.size // 2

# Create two high-density regions ("masses")
mass1_pos = (center - 8, center, center)
mass2_pos = (center + 8, center, center)

for pos in [mass1_pos, mass2_pos]:
    for dx in range(-3, 4):
        for dy in range(-3, 4):
            for dz in range(-3, 4):
                r = np.sqrt(dx**2 + dy**2 + dz**2)
                if r < 4:
                    mag = 5.0 * np.exp(-r**2 / 4)
                    universe.flux[pos[0]+dx, pos[1]+dy, pos[2]+dz, :] = mag

forces.calculate_density(universe)
print(f"Two mass concentrations created at x = {mass1_pos[0]} and x = {mass2_pos[0]}")

In [ ]:
# Visualize density field and gravitational force
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
z_slice = center

# Density field
density = universe.density[:, :, z_slice]
im0 = axes[0].imshow(density.T, origin='lower', cmap='hot')
axes[0].set_title('Density Field ρ', fontsize=12)
plt.colorbar(im0, ax=axes[0])

# Compute gravitational force field
grad_rho = discrete_gradient(universe.density)
F_grav = CONSTANTS.GRAVITY_BIAS * grad_rho

# Force magnitude
F_mag = np.linalg.norm(F_grav[:, :, z_slice, :], axis=-1)
im1 = axes[1].imshow(F_mag.T, origin='lower', cmap='viridis')
axes[1].set_title('|F_grav| = G_N |∇ρ|', fontsize=12)
plt.colorbar(im1, ax=axes[1])

# Force vectors (note: pointing toward density peaks)
step = 3
X, Y = np.meshgrid(np.arange(0, universe.size, step), np.arange(0, universe.size, step))
Fx = F_grav[::step, ::step, z_slice, 0]
Fy = F_grav[::step, ::step, z_slice, 1]

# Invert for attraction (force points toward mass, not away)
axes[2].quiver(X, Y, Fx.T, Fy.T, np.sqrt(Fx**2 + Fy**2).T, cmap='coolwarm')
axes[2].scatter([mass1_pos[0], mass2_pos[0]], [center, center], c='yellow', s=100, marker='*')
axes[2].set_title('Gravitational Force Field', fontsize=12)
axes[2].set_xlim(0, universe.size)
axes[2].set_ylim(0, universe.size)
axes[2].set_aspect('equal')

plt.suptitle('Gravity: Attraction via Density Gradient', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Electromagnetic-Like Forces

### Electric Force (Coulomb)
$$F_{elec} = -q \cdot \nabla \bar{q}$$

### Magnetic Force (Lorentz)
$$F_{mag} = \beta (\nabla \times J) \times \hat{J}$$

In [ ]:
# Create a charge distribution
universe = Universe(size=48)
center = universe.size // 2

# Positive charge at left
charge_pos = (center - 10, center, center)
# Negative charge at right  
charge_neg = (center + 10, center, center)

# Create charge field (stored in states for simplicity)
charge_field = np.zeros(universe.shape)

for pos, sign in [(charge_pos, 1.0), (charge_neg, -1.0)]:
    for dx in range(-3, 4):
        for dy in range(-3, 4):
            for dz in range(-3, 4):
                r = np.sqrt(dx**2 + dy**2 + dz**2)
                if r < 4:
                    charge_field[pos[0]+dx, pos[1]+dy, pos[2]+dz] = sign * np.exp(-r**2 / 4)

print(f"Charges created: + at x={charge_pos[0]}, - at x={charge_neg[0]}")

In [ ]:
# Visualize electric field
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
z_slice = center

# Charge distribution
vmax = abs(charge_field[:, :, z_slice]).max()
im0 = axes[0].imshow(charge_field[:, :, z_slice].T, origin='lower', 
                      cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('Charge Distribution (Red=+, Blue=-)', fontsize=12)
plt.colorbar(im0, ax=axes[0])

# Electric field = -gradient of charge field (like potential)
E_field = -discrete_gradient(charge_field)

# E-field magnitude
E_mag = np.linalg.norm(E_field[:, :, z_slice, :], axis=-1)
im1 = axes[1].imshow(E_mag.T, origin='lower', cmap='viridis')
axes[1].set_title('|E| = |∇q|', fontsize=12)
plt.colorbar(im1, ax=axes[1])

# E-field vectors
step = 3
X, Y = np.meshgrid(np.arange(0, universe.size, step), np.arange(0, universe.size, step))
Ex = E_field[::step, ::step, z_slice, 0]
Ey = E_field[::step, ::step, z_slice, 1]

axes[2].quiver(X, Y, Ex.T, Ey.T, np.sqrt(Ex**2 + Ey**2).T, cmap='coolwarm')
axes[2].scatter([charge_pos[0]], [center], c='red', s=100, marker='+', linewidths=3)
axes[2].scatter([charge_neg[0]], [center], c='blue', s=100, marker='_', linewidths=3)
axes[2].set_title('Electric Field Vectors', fontsize=12)
axes[2].set_xlim(0, universe.size)
axes[2].set_ylim(0, universe.size)
axes[2].set_aspect('equal')

plt.suptitle('Coulomb-Like Force: Opposite Charges Attract', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. The Curl and Magnetic-Like Effects

The **curl** of the flux field (∇×J) gives rise to magnetic-like behavior.

In [ ]:
# Create a vortex flux pattern (current loop analog)
universe = Universe(size=48)
center = universe.size // 2

# Circular current in XY plane
for x in range(universe.size):
    for y in range(universe.size):
        dx = x - center
        dy = y - center
        r = np.sqrt(dx**2 + dy**2) + 0.01
        
        if 3 < r < 15:
            # Tangential flux (CCW rotation)
            magnitude = 2.0 * np.exp(-(r - 8)**2 / 20)
            universe.flux[x, y, center, 0] = -magnitude * dy / r  # Jx = -sin(θ)
            universe.flux[x, y, center, 1] = magnitude * dx / r   # Jy = cos(θ)

print("Vortex flux pattern (current loop) created.")

In [ ]:
# Compute curl
curl_J = discrete_curl(universe.flux)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
z_slice = center

# Flux vectors
step = 2
X, Y = np.meshgrid(np.arange(0, universe.size, step), np.arange(0, universe.size, step))
Jx = universe.flux[::step, ::step, z_slice, 0]
Jy = universe.flux[::step, ::step, z_slice, 1]

axes[0].quiver(X, Y, Jx.T, Jy.T, np.sqrt(Jx**2 + Jy**2).T, cmap='viridis')
axes[0].set_title('Flux J (Current Loop)', fontsize=12)
axes[0].set_xlim(0, universe.size)
axes[0].set_ylim(0, universe.size)
axes[0].set_aspect('equal')

# Curl magnitude
curl_mag = np.linalg.norm(curl_J[:, :, z_slice, :], axis=-1)
im1 = axes[1].imshow(curl_mag.T, origin='lower', cmap='plasma')
axes[1].set_title('|∇×J| (Curl Magnitude)', fontsize=12)
plt.colorbar(im1, ax=axes[1])

# Curl z-component (like B_z from a current loop)
curl_z = curl_J[:, :, z_slice, 2]
vmax = abs(curl_z).max()
im2 = axes[2].imshow(curl_z.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[2].set_title('(∇×J)_z (Magnetic Field Analog)', fontsize=12)
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Curl Operator: Current Loop → Magnetic Field', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. The Yukawa (Strong-Like) Force

The strong force uses a **Yukawa potential** form:

$$F_{strong}(r) = g_s^2 \frac{e^{-m_\pi r}}{r^2}(1 + m_\pi r)$$

This is short-range, providing nuclear binding.

In [ ]:
# Yukawa potential comparison
r = np.linspace(0.1, 10, 200)

# Parameters
g_s = 1.0  # Strong coupling
m_pi = 0.5  # Pion mass (sets range)

# Yukawa force
F_yukawa = g_s**2 * np.exp(-m_pi * r) / r**2 * (1 + m_pi * r)

# Coulomb force (for comparison)
alpha = 0.00729
F_coulomb = alpha / r**2

plt.figure(figsize=(10, 6))
plt.semilogy(r, F_yukawa, 'r-', linewidth=2, label='Yukawa (Strong)')
plt.semilogy(r, F_coulomb, 'b--', linewidth=2, label='Coulomb (EM)')
plt.axvline(1/m_pi, color='gray', linestyle=':', label=f'Range = 1/m_π = {1/m_pi:.1f}')

plt.xlabel('Distance r', fontsize=12)
plt.ylabel('Force (log scale)', fontsize=12)
plt.title('Yukawa vs Coulomb: Short-Range vs Long-Range', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 10)
plt.ylim(1e-3, 100)
plt.show()

print(f"At r=1: Yukawa/Coulomb ratio = {F_yukawa[np.argmin(abs(r-1))]/F_coulomb[np.argmin(abs(r-1))]:.2f}")
print(f"At r=5: Yukawa/Coulomb ratio = {F_yukawa[np.argmin(abs(r-5))]/F_coulomb[np.argmin(abs(r-5))]:.2e}")

## 6. Force Competition: The Balance Point

Stable structures form when forces **balance**. Let's visualize the force landscape around a particle pair.

In [ ]:
# Create a proton-like system (three quarks)
universe = Universe(size=32)
center = universe.size // 2

# Three particles in a triangle (triad)
positions = [
    (center, center + 2, center),
    (center - 2, center - 1, center),
    (center + 2, center - 1, center),
]

for pos in positions:
    universe.states[pos] = 1  # Positive particles
    # Give each particle some flux
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                r = np.sqrt(dx**2 + dy**2 + dz**2) + 0.01
                if r < 3:
                    mag = 3.0 * np.exp(-r**2 / 2)
                    direction = np.array([dx, dy, dz]) / r
                    x, y, z = pos[0]+dx, pos[1]+dy, pos[2]+dz
                    if 0 <= x < universe.size and 0 <= y < universe.size and 0 <= z < universe.size:
                        universe.flux[x, y, z, :] += mag * direction

forces.calculate_density(universe)
print(f"Triad configuration created with 3 particles.")

In [ ]:
# Visualize the triad's field structure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
z_slice = center

# Density
im0 = axes[0].imshow(universe.density[:, :, z_slice].T, origin='lower', cmap='hot')
for pos in positions:
    axes[0].scatter([pos[0]], [pos[1]], c='cyan', s=100, marker='o', edgecolors='white')
axes[0].set_title('Density Field', fontsize=12)
plt.colorbar(im0, ax=axes[0])

# Divergence (shows sources)
div = discrete_divergence(universe.flux)
vmax = abs(div[:, :, z_slice]).max()
im1 = axes[1].imshow(div[:, :, z_slice].T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('Divergence ∇·J', fontsize=12)
plt.colorbar(im1, ax=axes[1])

# Force field (gradient of density)
grad_rho = discrete_gradient(universe.density)
step = 2
X, Y = np.meshgrid(np.arange(0, universe.size, step), np.arange(0, universe.size, step))
Fx = grad_rho[::step, ::step, z_slice, 0]
Fy = grad_rho[::step, ::step, z_slice, 1]

axes[2].quiver(X, Y, Fx.T, Fy.T, np.sqrt(Fx**2 + Fy**2).T, cmap='viridis')
for pos in positions:
    axes[2].scatter([pos[0]], [pos[1]], c='red', s=100, marker='o')
axes[2].set_title('Force Field ∇ρ', fontsize=12)
axes[2].set_xlim(0, universe.size)
axes[2].set_ylim(0, universe.size)
axes[2].set_aspect('equal')

plt.suptitle('Triad Force Structure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Gauge Symmetry: U(1) Emergence

FTD argues that **U(1) gauge symmetry** emerges from the constraint structure.

The Helmholtz decomposition: $J = J_T + J_L$
- $J_T$ (transverse): $\nabla \cdot J_T = 0$
- $J_L$ (longitudinal): $\nabla \times J_L = 0$

The longitudinal part is **constrained** by charge conservation (Gauss law), leaving **2 physical degrees of freedom** - matching the 2 polarizations of photons.

In [ ]:
# Demonstrate transverse vs longitudinal decomposition
universe = Universe(size=32)
center = universe.size // 2

# Create a mix of transverse and longitudinal flux
for x in range(universe.size):
    for y in range(universe.size):
        dx = x - center
        dy = y - center
        r = np.sqrt(dx**2 + dy**2) + 0.01
        
        if r < 12:
            # Transverse (rotational) component
            trans_mag = np.exp(-r**2 / 30)
            J_trans = np.array([-dy/r, dx/r, 0]) * trans_mag
            
            # Longitudinal (radial) component
            long_mag = 0.5 * np.exp(-r**2 / 30)
            J_long = np.array([dx/r, dy/r, 0]) * long_mag
            
            universe.flux[x, y, center, :] = J_trans + J_long

In [ ]:
# Analyze the field
curl_J = discrete_curl(universe.flux)
div_J = discrete_divergence(universe.flux)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
z_slice = center

# Total flux
step = 2
X, Y = np.meshgrid(np.arange(0, universe.size, step), np.arange(0, universe.size, step))
Jx = universe.flux[::step, ::step, z_slice, 0]
Jy = universe.flux[::step, ::step, z_slice, 1]
axes[0].quiver(X, Y, Jx.T, Jy.T, np.sqrt(Jx**2 + Jy**2).T, cmap='viridis')
axes[0].set_title('Total J = J_T + J_L', fontsize=12)
axes[0].set_aspect('equal')

# Curl (detects rotational/transverse)
curl_z = curl_J[:, :, z_slice, 2]
vmax = abs(curl_z).max()
im1 = axes[1].imshow(curl_z.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('∇×J (Transverse Signature)', fontsize=12)
plt.colorbar(im1, ax=axes[1])

# Divergence (detects longitudinal)
vmax = abs(div_J[:, :, z_slice]).max()
im2 = axes[2].imshow(div_J[:, :, z_slice].T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[2].set_title('∇·J (Longitudinal Signature)', fontsize=12)
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Helmholtz Decomposition: Transverse vs Longitudinal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Count degrees of freedom
print("\nDegrees of Freedom Analysis:")
print(f"  J has 3 components")
print(f"  Gauss constraint (∇·J ~ ρ) removes 1")
print(f"  Remaining: 2 physical (transverse) modes")
print(f"  → Matches 2 photon polarizations!")

## 8. Summary: Forces from Fields

### The Discrete Operators

| Operator | Symbol | Role |
|----------|--------|------|
| Gradient | ∇ | Direction of maximum change |
| Divergence | ∇· | Source/sink detection |
| Curl | ∇× | Rotation detection |
| Laplacian | ∇² | Wave propagation |

### The Four Forces

| Force | Formula | Character |
|-------|---------|----------|
| Gravity | F = G∇ρ | Long-range, attractive |
| Electric | F = -q∇q | Long-range, ± |
| Magnetic | F = (∇×J)×Ĵ | Velocity-dependent |
| Strong | F = Yukawa | Short-range, binding |

### Key Insights

1. All forces come from **gradients of fields**
2. **Gauge symmetry** (U(1)) emerges from constraint structure
3. **2 polarizations** = 3 components - 1 constraint
4. Stable structures form at **force balance** points

**Next**: See `04_binding_structures.ipynb` for how forces create stable particles.